# Exploratory Analysis of New York City Airbnb Listings

**Author:** Marius Jochheim  
**Dataset:** New York City Airbnb Open Data (2019)

This project develops a reproducible Python workflow for cleaning, exploring,
and visualizing New York City Airbnb listing data. The analysis focuses on how
listing prices and availability differ across neighbourhood groups and room
types.

## Analysis Questions

This analysis investigates the following questions:

1. How do Airbnb listing prices vary across New York City neighbourhood groups?
2. How do prices differ between room types?
3. How does listing availability vary across neighbourhood groups and room types?
4. Are price, review activity, minimum stay, and availability meaningfully related?

# Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset

In [2]:
DATA_PATH = "dataset/AB_NYC_2019.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
print(f"The dataset contains {df.shape[0]:,} rows and {df.shape[1]} columns.")

The dataset contains 48,895 rows and 16 columns.


# Initial Data Inspection

Before cleaning the dataset, its structure, data types, missing values,
duplicates, and numerical distributions are inspected.

## Columns, missing values and duplicates

In [4]:
def inspect_dataset(data):
    """
    Display a concise overview of a pandas DataFrame.

    The function reports the dataset dimensions, data types, missing-value
    counts, duplicated rows, and descriptive statistics for numerical columns.

    Parameters
    ----------
    data : pandas.DataFrame
        The dataset to inspect.

    Returns
    -------
    None
        The function prints and displays inspection results.
    """
    print(f"Rows: {data.shape[0]:,}")
    print(f"Columns: {data.shape[1]}")
    print(f"Duplicated rows: {data.duplicated().sum():,}")

    display(
        data.dtypes
        .astype(str)
        .rename("data_type")
        .to_frame()
    )

    missing_summary = (
        df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
    )

    missing_summary["missing_percentage"] = (
        missing_summary["missing_count"] / len(df) * 100
    )

    missing_summary = missing_summary.sort_values(
        by="missing_count",
        ascending=False
    )

    display(
        missing_summary
    )

    display(data.describe(include="number").T)

In [5]:
inspect_dataset(df)

Rows: 48,895
Columns: 16
Duplicated rows: 0


,data_type
id,int64
name,str
host_id,int64
host_name,str
neighbourhood_group,str
neighbourhood,str
latitude,float64
longitude,float64
room_type,str
price,int64


,missing_count,missing_percentage
last_review,10052,20.558339
reviews_per_month,10052,20.558339
host_name,21,0.042949
name,16,0.032723
neighbourhood_group,0,0.000000
neighbourhood,0,0.000000
id,0,0.000000
host_id,0,0.000000
longitude,0,0.000000
latitude,0,0.000000


,count,mean,std,min,25%,50%,75%,max
id,48895.0,1.901714e+07,1.098311e+07,2539.00000,9.471945e+06,1.967728e+07,2.915218e+07,3.648724e+07
host_id,48895.0,6.762001e+07,7.861097e+07,2438.00000,7.822033e+06,3.079382e+07,1.074344e+08,2.743213e+08
latitude,48895.0,4.072895e+01,5.453008e-02,40.49979,4.069010e+01,4.072307e+01,4.076311e+01,4.091306e+01
longitude,48895.0,-7.395217e+01,4.615674e-02,-74.24442,-7.398307e+01,-7.395568e+01,-7.393627e+01,-7.371299e+01
price,48895.0,1.527207e+02,2.401542e+02,0.00000,6.900000e+01,1.060000e+02,1.750000e+02,1.000000e+04
minimum_nights,48895.0,7.029962e+00,2.051055e+01,1.00000,1.000000e+00,3.000000e+00,5.000000e+00,1.250000e+03
number_of_reviews,48895.0,2.327447e+01,4.455058e+01,0.00000,1.000000e+00,5.000000e+00,2.400000e+01,6.290000e+02
reviews_per_month,38843.0,1.373221e+00,1.680442e+00,0.01000,1.900000e-01,7.200000e-01,2.020000e+00,5.850000e+01
calculated_host_listings_count,48895.0,7.143982e+00,3.295252e+01,1.00000,1.000000e+00,1.000000e+00,2.000000e+00,3.270000e+02
availability_365,48895.0,1.127813e+02,1.316223e+02,0.00000,0.000000e+00,4.500000e+01,2.270000e+02,3.650000e+02


## Categorical Values

In [6]:
for column in ["neighbourhood_group", "room_type"]:
    print(f"\nValues in {column}:")
    print(df[column].value_counts(dropna=False))


Values in neighbourhood_group:
neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64

Values in room_type:
room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64


In [7]:
categorical_summary = pd.DataFrame({
    "unique_values": df[
        ["neighbourhood_group", "neighbourhood", "room_type"]
    ].nunique(),
    "missing_values": df[
        ["neighbourhood_group", "neighbourhood", "room_type"]
    ].isna().sum()
})

categorical_summary

,unique_values,missing_values
neighbourhood_group,5,0
neighbourhood,221,0
room_type,3,0


## Numerical Ranges

In [8]:
analysis_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

df[analysis_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
price,48895.0,152.720687,240.154170,0.00,69.00,106.00,175.00,10000.0
minimum_nights,48895.0,7.029962,20.510550,1.00,1.00,3.00,5.00,1250.0
number_of_reviews,48895.0,23.274466,44.550582,0.00,1.00,5.00,24.00,629.0
reviews_per_month,38843.0,1.373221,1.680442,0.01,0.19,0.72,2.02,58.5
calculated_host_listings_count,48895.0,7.143982,32.952519,1.00,1.00,1.00,2.00,327.0
availability_365,48895.0,112.781327,131.622289,0.00,0.00,45.00,227.00,365.0


## Initial Inspection Observations

- The dataset contains **48,895 rows** and **16 columns**.
- Each row represents an Airbnb listing in New York City.
- Missing values occur primarily in `last_review`	and `reviews_per_month`.
- The `last_review` column is currently stored as text and should be converted
  to a date type.
- The complete dataset contains **no duplicated rows**.
- The price and minimum-night variables contain large values that require
  further investigation before deciding whether they should be retained,
  filtered, or treated as outliers.